# NDJF Event Diagnostic Atlas

This notebook is the dedicated plots framework for subjective event diagnosis. It is separate from the later feature-table and clustering workflow.

Use it to inspect:
- convergence (`del dot u`) and low-level flow
- cloud proxy / OLR if available
- geopotential-height structure (`Z850` / `Z700`)
- circulation structure (`del cross u` / relative vorticity)
- frontality via `|grad T|`

Important note:
- the geopotential panels in this notebook default to **domain-relative** height departures for fast event diagnosis
- the later objective subtype notebook can compute a more formal climatological anomaly index for clustering/features

In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = True
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import BoundingBox, EXTENDED_DOMAIN, JPCZ_POLYGON_VERTICES, VORTICITY_BOX, WORKING_DOMAIN
from jpcz_catalog.detect import compute_divergence_field
from jpcz_catalog.diagnostics import (
    compute_domain_relative_anomaly,
    compute_geopotential_height_field,
    compute_relative_vorticity_field,
    compute_temperature_gradient_magnitude,
    load_offset_snapshot,
    load_snapshot,
)
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.plotting import plot_event_peak_quicklook, plot_scalar_field_map

REVIEW_CATALOG_PATH = Path(DRIVE_OUTPUT_DIR) / "jpcz_catalog_ndjf_merged_12h_manual_verification.csv"
OUTPUT_DIR = Path("outputs/verification/event_diagnostic_atlas")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVENT_INDEX = 0
CLOUD_VARIABLE = None
CLOUD_PANEL_MODE = "side_by_side"
SAVE_PLOTS = False

SYNOPTIC_LEVEL = 850
SECONDARY_LEVEL = 700
TIME_OFFSETS_HOURS = (-12, 0, 12)

# Editable diagnostic boxes for subjective and later objective analysis.
COASTAL_JAPAN_BOX = BoundingBox(lon_min=129.0, lon_max=141.0, lat_min=34.5, lat_max=39.8)
PACIFIC_EAST_OF_JAPAN_BOX = BoundingBox(lon_min=141.0, lon_max=150.0, lat_min=33.0, lat_max=42.5)
HOKKAIDO_BOX = BoundingBox(lon_min=138.0, lon_max=146.0, lat_min=41.0, lat_max=47.5)
SEA_OF_JAPAN_BOX = BoundingBox(lon_min=129.0, lon_max=138.5, lat_min=35.0, lat_max=43.5)

PLOT_BOXES = {
    "Coastal Japan": COASTAL_JAPAN_BOX,
    "Pacific east of Japan": PACIFIC_EAST_OF_JAPAN_BOX,
    "Hokkaido": HOKKAIDO_BOX,
    "Sea of Japan": SEA_OF_JAPAN_BOX,
    "Shinoda vorticity box": VORTICITY_BOX,
}

def maybe_copy_to_drive(path: Path):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return
    drive_dir = Path(DRIVE_OUTPUT_DIR) / OUTPUT_DIR.name
    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, drive_dir / path.name)
    print("Copied to Drive:", drive_dir / path.name)

def build_synoptic_title(prefix: str, event_row: pd.Series, offset_hours: int, level: int) -> str:
    valid_time_utc = pd.Timestamp(event_row["event_peak"]) + pd.Timedelta(hours=offset_hours)
    valid_time_jst = valid_time_utc + pd.Timedelta(hours=9)
    return (
        f"{prefix} | idx {event_row.name:03d} | {level} hPa | offset {offset_hours:+d} h\n"
        f"UTC {valid_time_utc:%Y-%m-%d %H:%M} | JST {valid_time_jst:%Y-%m-%d %H:%M}"
    )


In [ ]:
review_df = pd.read_csv(
    REVIEW_CATALOG_PATH,
    parse_dates=["event_start", "event_end", "event_peak"],
)
review_df = add_japan_local_time_columns(review_df)

event_row = review_df.loc[EVENT_INDEX].copy()
peak_time_utc = pd.Timestamp(event_row["event_peak"])
peak_time_jst = pd.Timestamp(event_row["event_peak_jst"])

print("Selected event index:", EVENT_INDEX)
display(
    review_df.loc[[EVENT_INDEX], [
        "event_start",
        "event_end",
        "event_peak",
        "event_peak_jst",
        "duration_hours",
        "event_peak_D_1e5_s-1",
        "verified_event",
        "position_group_manual",
        "pattern_type_manual",
        "verification_notes",
    ]]
)


In [ ]:
ds = open_arco_era5()

candidate_keywords = ("cloud", "olr", "radiation", "toa", "brightness", "longwave", "shortwave")
candidate_vars = [name for name in ds.data_vars if any(keyword in name.lower() for keyword in candidate_keywords)]
print("Possible cloud/OLR-like variables in this ERA5 store:")
print(candidate_vars)

if CLOUD_VARIABLE is not None and CLOUD_VARIABLE not in ds.data_vars:
    raise ValueError(f"CLOUD_VARIABLE '{CLOUD_VARIABLE}' not found in this ERA5 store.")


In [ ]:
peak_variables = ["u_component_of_wind", "v_component_of_wind"]
if CLOUD_VARIABLE is not None:
    peak_variables.append(CLOUD_VARIABLE)

peak_snapshot_925 = load_snapshot(
    ds,
    peak_time_utc,
    variables=peak_variables,
    domain=WORKING_DOMAIN,
    level=925,
)
peak_divergence_925 = compute_divergence_field(peak_snapshot_925)
peak_cloud_field = peak_snapshot_925[CLOUD_VARIABLE] if CLOUD_VARIABLE is not None else None

quicklook_title = (
    f"NDJF event {EVENT_INDEX} | peak UTC {peak_time_utc:%Y-%m-%d %H:%M} | JST {peak_time_jst:%Y-%m-%d %H:%M}\n"
    f"peak D={event_row['event_peak_D_1e5_s-1']:.2f} | duration={int(event_row['duration_hours'])} h | pattern={event_row.get('pattern_type_manual', '')}"
)
quicklook_path = OUTPUT_DIR / f"event_{EVENT_INDEX:03d}_peak_quicklook.png"

plot_event_peak_quicklook(
    peak_snapshot_925,
    peak_divergence_925,
    domain=WORKING_DOMAIN,
    polygon_vertices=JPCZ_POLYGON_VERTICES,
    vorticity_box=VORTICITY_BOX,
    title=quicklook_title,
    cloud_field=peak_cloud_field,
    cloud_label=CLOUD_VARIABLE if CLOUD_VARIABLE is not None else "Cloud proxy",
    max_location=(event_row["peak_max_convergence_lat"], event_row["peak_max_convergence_lon"]),
    centroid_location=(event_row["peak_convergence_centroid_lat"], event_row["peak_convergence_centroid_lon"]),
    cloud_panel_mode=CLOUD_PANEL_MODE,
    save_path=quicklook_path if SAVE_PLOTS else None,
)

if SAVE_PLOTS:
    maybe_copy_to_drive(quicklook_path)


In [ ]:
for offset_hours in TIME_OFFSETS_HOURS:
    synoptic_snapshot = load_offset_snapshot(
        ds,
        peak_time_utc,
        offset_hours=offset_hours,
        variables=["u_component_of_wind", "v_component_of_wind", "geopotential"],
        domain=EXTENDED_DOMAIN,
        level=SYNOPTIC_LEVEL,
    )
    z_field = compute_geopotential_height_field(synoptic_snapshot)
    z_domain_relative = compute_domain_relative_anomaly(z_field)

    z_levels = np.arange(
        np.floor(float(z_field.min()) / 4.0) * 4.0,
        np.ceil(float(z_field.max()) / 4.0) * 4.0 + 4.0,
        4.0,
    )
    z_path = OUTPUT_DIR / f"event_{EVENT_INDEX:03d}_z{SYNOPTIC_LEVEL}_offset_{offset_hours:+d}h.png"

    plot_scalar_field_map(
        z_domain_relative,
        domain=EXTENDED_DOMAIN,
        title=build_synoptic_title("Domain-relative geopotential height", event_row, offset_hours, SYNOPTIC_LEVEL),
        colorbar_label=f"{SYNOPTIC_LEVEL} hPa geopotential height departure (gpm)",
        polygon_vertices=JPCZ_POLYGON_VERTICES,
        boxes=PLOT_BOXES,
        vector_snapshot=synoptic_snapshot,
        contour_field=z_field,
        contour_levels=z_levels,
        levels=13,
        cmap="RdBu_r",
        save_path=z_path if SAVE_PLOTS else None,
    )

    if SAVE_PLOTS:
        maybe_copy_to_drive(z_path)


In [ ]:
for offset_hours in TIME_OFFSETS_HOURS:
    vort_snapshot = load_offset_snapshot(
        ds,
        peak_time_utc,
        offset_hours=offset_hours,
        variables=["u_component_of_wind", "v_component_of_wind"],
        domain=EXTENDED_DOMAIN,
        level=925,
    )
    zeta_field = compute_relative_vorticity_field(vort_snapshot)
    zeta_display = zeta_field * 1e5
    zeta_path = OUTPUT_DIR / f"event_{EVENT_INDEX:03d}_zeta925_offset_{offset_hours:+d}h.png"

    plot_scalar_field_map(
        zeta_display,
        domain=EXTENDED_DOMAIN,
        title=build_synoptic_title("Relative vorticity", event_row, offset_hours, 925),
        colorbar_label="925 hPa relative vorticity (1e-5 s^-1)",
        polygon_vertices=JPCZ_POLYGON_VERTICES,
        boxes=PLOT_BOXES,
        vector_snapshot=vort_snapshot,
        levels=np.arange(-12, 13, 1),
        cmap="PuOr_r",
        save_path=zeta_path if SAVE_PLOTS else None,
    )

    if SAVE_PLOTS:
        maybe_copy_to_drive(zeta_path)


In [ ]:
for offset_hours in TIME_OFFSETS_HOURS:
    frontal_snapshot = load_offset_snapshot(
        ds,
        peak_time_utc,
        offset_hours=offset_hours,
        variables=["u_component_of_wind", "v_component_of_wind", "temperature", "geopotential"],
        domain=EXTENDED_DOMAIN,
        level=SYNOPTIC_LEVEL,
    )
    grad_t_field = compute_temperature_gradient_magnitude(frontal_snapshot)
    grad_t_display = grad_t_field * float(grad_t_field.attrs.get("display_scale_factor", 1.0))
    z_field = compute_geopotential_height_field(frontal_snapshot)
    z_levels = np.arange(
        np.floor(float(z_field.min()) / 4.0) * 4.0,
        np.ceil(float(z_field.max()) / 4.0) * 4.0 + 4.0,
        4.0,
    )
    grad_path = OUTPUT_DIR / f"event_{EVENT_INDEX:03d}_gradT{SYNOPTIC_LEVEL}_offset_{offset_hours:+d}h.png"

    plot_scalar_field_map(
        grad_t_display,
        domain=EXTENDED_DOMAIN,
        title=build_synoptic_title("Temperature-gradient magnitude", event_row, offset_hours, SYNOPTIC_LEVEL),
        colorbar_label=f"|grad T| at {SYNOPTIC_LEVEL} hPa (K per 100 km)",
        polygon_vertices=JPCZ_POLYGON_VERTICES,
        boxes=PLOT_BOXES,
        vector_snapshot=frontal_snapshot,
        contour_field=z_field,
        contour_levels=z_levels,
        levels=12,
        cmap="magma",
        extend="max",
        save_path=grad_path if SAVE_PLOTS else None,
    )

    if SAVE_PLOTS:
        maybe_copy_to_drive(grad_path)


## MODIS Satellite Truth Panel

This section pulls true-color MODIS imagery through NASA GIBS for the event date.

Use it as a practical historical satellite truth layer for the full record.

Important limitations:
- this is daily MODIS imagery, not exact hourly geostationary imagery
- it is still very useful for checking cloud-band structure and broad event context
- Himawari-style hourly geostationary imagery is a later add-on, mainly for the modern subset

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

from jpcz_catalog.satellite import (
    build_gibs_wms_getmap_url,
    default_modis_layers_for_date,
    download_gibs_image,
    layer_available_for_date,
)

SATELLITE_DOMAIN = EXTENDED_DOMAIN
SATELLITE_WIDTH = 1400
SATELLITE_HEIGHT = 1000
SATELLITE_IMAGE_FORMAT = "image/jpeg"
SATELLITE_LAYERS = None  # e.g. ['MODIS_Terra_CorrectedReflectance_TrueColor']

satellite_date = peak_time_utc.normalize()
satellite_layers = SATELLITE_LAYERS or default_modis_layers_for_date(satellite_date)
downloaded_satellite = []

for layer_name in satellite_layers:
    if not layer_available_for_date(layer_name, satellite_date):
        print(f"Skipping unavailable layer for {satellite_date:%Y-%m-%d}: {layer_name}")
        continue

    satellite_url = build_gibs_wms_getmap_url(
        layer_name=layer_name,
        target_date=satellite_date,
        bbox=SATELLITE_DOMAIN,
        width=SATELLITE_WIDTH,
        height=SATELLITE_HEIGHT,
        image_format=SATELLITE_IMAGE_FORMAT,
    )
    layer_slug = (
        layer_name.replace("MODIS_", "")
        .replace("_CorrectedReflectance_TrueColor", "")
        .lower()
    )
    output_path = OUTPUT_DIR / f"event_{EVENT_INDEX:03d}_{layer_slug}_{satellite_date:%Y%m%d}.jpg"

    try:
        download_gibs_image(satellite_url, output_path)
        downloaded_satellite.append((layer_name, satellite_url, output_path))
        print(f"Downloaded {layer_name}: {output_path}")
        print(satellite_url)
        if SAVE_PLOTS:
            maybe_copy_to_drive(output_path)
    except Exception as exc:
        print(f"Failed to download {layer_name}: {exc}")

if downloaded_satellite:
    fig, axes = plt.subplots(1, len(downloaded_satellite), figsize=(8 * len(downloaded_satellite), 8))
    if len(downloaded_satellite) == 1:
        axes = [axes]

    for ax, (layer_name, satellite_url, output_path) in zip(axes, downloaded_satellite):
        ax.imshow(Image.open(output_path))
        ax.axis("off")
        pretty_name = layer_name.replace("MODIS_", "").replace("_CorrectedReflectance_TrueColor", " true color")
        ax.set_title(pretty_name)

    fig.suptitle(
        f"MODIS daily true-color imagery | event {EVENT_INDEX} | date {satellite_date:%Y-%m-%d}\n"
        f"event peak UTC {peak_time_utc:%Y-%m-%d %H:%M} | JST {peak_time_jst:%Y-%m-%d %H:%M}\n"
        "Daily imagery is not exact hourly geostationary timing, but is useful for cloud-band truth checking.",
        y=0.98,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    plt.show()
else:
    print("No MODIS satellite panels were downloaded for this event date.")


## Cross-section Note

Cross sections belong in this diagnostic workflow, but not as the first mass-processing step for all `201` events.

Recommended use:
- pick representative cases from each subjective subtype
- choose the cross-section line after seeing the map geometry
- use cross sections later for frontal structure, vertical motion, and jet interpretation

That keeps the atlas notebook fast enough for broad event review, while still leaving room for deeper case studies.